# WA Fire History and Risk: Data-Center Sites

Walkthrough behind `/reports/wa-fire-risk`. Two real layers, not one invented
temperature-and-history composite: MTBS's actual historical burn perimeters (1984-2024) and the
U.S. Forest Service's own Wildfire Hazard Potential (agency-scored hazard raster), each joined
against real tracked WA data-center sites with a plain `geopandas.sjoin`.

Production version with assertions and the JSON this notebook's numbers should match:
`scripts/analysis_wa_fire_risk.py`.


## Setup

In [ ]:
import importlib.util
import json
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import Point

from pathlib import Path

DATA = Path("../data").resolve()
SHARED = DATA / "shared"


def _load_script(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


## 1. Download -- MTBS fire perimeters

MTBS ships one national burned-area boundary shapefile (~390 MB). `fetch_mtbs` (from
`export_wa_fire_perimeters.py`) streams it to disk, clips to WA, and caches the raw clip to
`data/shared/wa_fire_perimeters_raw.geojson` -- re-running this cell reuses that cache.


In [ ]:
export_perimeters = _load_script("export_perimeters", "../scripts/export_wa_fire_perimeters.py")
wa_raw = export_perimeters.fetch_mtbs()

print(f"{len(wa_raw)} MTBS fire perimeters intersecting WA")
print(f"columns: {list(wa_raw.columns)}")
wa_raw.head(3)


## 2. Open the data -- reduce to what we need

MTBS's raw columns aren't self-explanatory (`ig_date`, `burnbndac`, `asmnt_type`). Note:
`asmnt_type` is MTBS's own mapping-methodology label (Initial/Extended/Comprehensive) -- **not**
a burn-severity class; the boundary product has no single per-fire severity field.


In [ ]:
wa_fires = wa_raw.copy()
wa_fires["year"] = pd.to_datetime(wa_fires["ig_date"], errors="coerce").dt.year
wa_fires = wa_fires.rename(columns={"burnbndac": "acres", "incid_name": "name",
                                     "asmnt_type": "assessment_type"})
wa_fires = wa_fires[["name", "year", "acres", "assessment_type", "geometry"]]

print(f"{len(wa_fires)} fires, {wa_fires['year'].min():.0f}-{wa_fires['year'].max():.0f}")
wa_fires.sort_values("acres", ascending=False).head(10)[["name", "year", "acres"]]


## 3. Map -- fire perimeters by decade

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
wa_fires.plot(column="year", cmap="YlOrRd", legend=True, ax=ax, edgecolor="none")
ax.set_title(f"MTBS-mapped fire perimeters, WA, {wa_fires['year'].min():.0f}-{wa_fires['year'].max():.0f}")
ax.set_axis_off()
plt.show()


## 4. Download -- USFS Wildfire Hazard Potential

WHP's download bundles a File Geodatabase (needs GDAL -- off-limits here) but *also* ships plain
GeoTIFF exports with `.tfw` worldfiles alongside it. `fetch_whp` + `crop_to_wa` (from
`fetch_whp_wa.py`) read the GeoTIFF with PIL + a hand-parsed worldfile + pyproj, no rasterio
needed, and cache the WA crop to `data/shared/whp_wa.npy`. Skip the download entirely if that
cache already exists.


In [ ]:
whp_npy_path = os.path.join(SHARED, "whp_wa.npy")
whp_transform_path = os.path.join(SHARED, "whp_wa_transform.json")

if os.path.exists(whp_npy_path) and os.path.exists(whp_transform_path):
    print("Using cached WHP crop:", whp_npy_path)
    whp_arr = np.load(whp_npy_path)
    whp_transform = json.load(open(whp_transform_path))
else:
    fetch_whp_mod = _load_script("fetch_whp_mod", "../scripts/fetch_whp_wa.py")
    tif_path, tfw_path = fetch_whp_mod.fetch_whp()
    whp_arr, whp_transform = fetch_whp_mod.crop_to_wa(tif_path, tfw_path)

print(f"WHP array: {whp_arr.shape} pixels, 270m native resolution, CRS={whp_transform['crs']}")
print("Pixel values: 1-5 = hazard class (very low..very high), 6 = non-burnable, "
      "7 = water, 255 = nodata")


## 5. Map -- hazard potential

A static `imshow` in the raster's own native Albers CRS is geometrically correct as-is -- no
reprojection needed for a one-off matplotlib figure. (The *interactive* Leaflet version of this
map needed real care here: `L.imageOverlay` stretches images linearly in Web Mercator space, not
linearly by latitude, which took three real bugs to get right -- see
`scripts/render_whp_png.py`'s docstring. A static plot in native CRS sidesteps that class of bug
entirely.)


In [ ]:
extent = [
    whp_transform["origin_x"],
    whp_transform["origin_x"] + whp_arr.shape[1] * whp_transform["pixel_x"],
    whp_transform["origin_y"] + whp_arr.shape[0] * whp_transform["pixel_y"],
    whp_transform["origin_y"],
]
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(whp_arr, extent=extent, cmap="RdYlGn_r", vmin=1, vmax=7)
ax.set_title("USFS Wildfire Hazard Potential -- native 270m Albers grid")
plt.colorbar(im, ax=ax, label="1=very low .. 5=very high, 6=non-burnable, 7=water")
plt.show()


## 6. Data-center sites -- load and join to fire history

Real tracked WA sites, confirmed by 2+ independent sources. The join to fire perimeters is a
plain `geopandas.sjoin` (`predicate="within"`) -- does a site's real coordinate fall inside any
real MTBS burn perimeter.


In [ ]:
dc_gdf = gpd.read_file(f"{SHARED}/tracked_datacenters.geojson")
dc_wa = dc_gdf[(dc_gdf["state"] == "WA") & (dc_gdf["confirmed"])].to_crs(wa_fires.crs)
print(f"{len(dc_wa)} confirmed WA data-center sites")

joined = gpd.sjoin(
    dc_wa, wa_fires[["name", "year", "geometry"]],
    predicate="within", how="left", lsuffix="dc", rsuffix="fire",
)
joined[["name_dc", "operator", "city", "name_fire", "year"]].rename(
    columns={"name_dc": "site", "name_fire": "burned_in"}
)


## 7. Sample WHP hazard class at each DC site

Point sampling only -- never a polygon centroid (a naive vertex-averaged centroid can land
outside an irregular/concave fire perimeter or on unrepresentative ground; confirmed by spot-check
during the production script's development). DC sites are real point coordinates, so this
problem doesn't apply to them -- only to any future per-perimeter WHP summary.


In [ ]:
WHP_LABELS = {1: "very low", 2: "low", 3: "moderate", 4: "high", 5: "very high",
              6: "non-burnable", 7: "water", 255: None}
to_whp = Transformer.from_crs("EPSG:4326", whp_transform["crs"], always_xy=True)


def whp_class_at(lon, lat):
    x, y = to_whp.transform(lon, lat)
    col = int(round((x - whp_transform["origin_x"]) / whp_transform["pixel_x"]))
    row = int(round((whp_transform["origin_y"] - y) / -whp_transform["pixel_y"]))
    if 0 <= row < whp_arr.shape[0] and 0 <= col < whp_arr.shape[1]:
        return int(whp_arr[row, col])
    return None


rows = []
for _, site in dc_wa.iterrows():
    cls = whp_class_at(site.geometry.x, site.geometry.y)
    rows.append({"site": site["name"], "city": site["city"], "whp_class": cls,
                 "whp_label": WHP_LABELS.get(cls)})
pd.DataFrame(rows)


## 8. Findings -- statewide burned share and hazard share

Area-weighted via Monte Carlo sampling over WA's real boundary (not the raster's padded bounding
box) -- the same method the production script uses, at a smaller sample size here for a quick
notebook run.


In [ ]:
wa_boundary = gpd.read_file(f"{DATA}/WA/raw/state.geojson")
wa_union = wa_boundary.union_all()
minlon, minlat, maxlon, maxlat = wa_boundary.total_bounds

rng = np.random.default_rng(0)
n_target = 5_000
lons, lats = np.empty(0), np.empty(0)
while lons.size < n_target:
    cand_lon = rng.uniform(minlon, maxlon, n_target * 2)
    cand_lat = rng.uniform(minlat, maxlat, n_target * 2)
    keep = np.array([wa_union.contains(Point(x, y)) for x, y in zip(cand_lon, cand_lat)])
    lons = np.concatenate([lons, cand_lon[keep]])
    lats = np.concatenate([lats, cand_lat[keep]])
lons, lats = lons[:n_target], lats[:n_target]

perim_union = wa_fires.geometry.union_all()
sample_pts = gpd.GeoSeries([Point(x, y) for x, y in zip(lons, lats)], crs="EPSG:4326")
burned = sample_pts.within(perim_union)
classes = np.array([whp_class_at(lon, lat) for lon, lat in zip(lons, lats)])

print(f"Monte Carlo sample: n={n_target}")
print(f"  {100 * burned.mean():.1f}% of WA land area has burned at least once since "
      f"{wa_fires['year'].min():.0f}")
print(f"  {100 * (classes == 4).mean():.1f}% of WA reads WHP 'high' hazard potential")
print(f"  {100 * (classes == 5).mean():.1f}% of WA reads WHP 'very high' hazard potential")


## Caveats

- **WHP's fuel model reads agricultural/CRP land as "non-burnable" by definition** -- including
  the shrub-steppe/grassland that carried the 2020 Pearl Hill and Cold Springs fires (187,000 -
  218,000 acres each). A "non-burnable" or "low" reading is a fuel-model read, not a guarantee
  against grass-fire risk.
- **`assessment_type` is not a severity class** -- see section 2 above.
- **MTBS's fire-name identification is not exhaustive.** Large multi-fire complexes are sometimes
  mapped under their original component-fire names rather than a merged complex name.


## Reproduce

- Numbers with assertions: `python3 scripts/analysis_wa_fire_risk.py`
- Fire perimeter export: `scripts/export_wa_fire_perimeters.py`
- WHP fetch + crop: `scripts/fetch_whp_wa.py`
- Live page: https://datacenters.simonhansedasi.com/reports/wa-fire-risk
